In [15]:
# Import

import os
import random
import numpy as np
import torch
import pandas as pd
import transformers
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import DataCollatorWithPadding

In [16]:
# Setup, reproducibility, device
SEED = 123

def set_all_seeds(seed: int = SEED):
    # Fix all RNG sources: high variance with ~137 training docs, repeatability matters
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # no-op if no CUDA

set_all_seeds(SEED)

# Deterministic cuDNN: slightly slower, reproducible results (negligible cost on small data)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Auto device detection: CUDA -> MPS -> CPU
def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"[device] CUDA active -> {torch.cuda.get_device_name(0)}")
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("[device] Apple MPS active")
    else:
        dev = torch.device("cpu")
        print("[device] No GPU found -> CPU (BERT will be slow)")
    return dev

device = get_device()



[device] CUDA active -> NVIDIA GeForce RTX 5070


In [17]:
# Sanity check: 'CUDA build=None' means a CPU-only torch is installed
print(f"[versions] torch={torch.__version__} | CUDA build={torch.version.cuda}")
print(f"[seed] {SEED}")

[versions] torch=2.12.1+cu130 | CUDA build=13.0
[seed] 123


In [18]:
# Load data, select columns, integrity checks, token-length profile

CSV_PATH = "..\Dati\Processed\dataset_processed_quantile1_sentences.csv"  
MODEL_NAME = "bert-base-cased"                            # cased: casing/punct are stylistic signal
KEEP_COLS = ["article_id", "topic_id", "binary_label", "fold", "text_bert"]

# Load 
df = pd.read_csv(CSV_PATH)
df = df[KEEP_COLS].copy()
df = df.reset_index(drop=True)  # avoid __index_level_0__ leaking into HF Dataset later

# Integrity checks 
print(f"[shape] {df.shape[0]} docs, {df.shape[1]} cols")
print(f"[NaN]\n{df.isna().sum()}")
print(f"[dup article_id] {df['article_id'].duplicated().sum()}")
print(f"[folds] {sorted(df['fold'].unique())}")

# Class balance overall and per fold (must stay ~2:1)
print("\n[label balance overall]")
print(df["binary_label"].value_counts(normalize=True).round(3).to_dict())
print("\n[label balance per fold]")
print(df.groupby("fold")["binary_label"].value_counts(normalize=True).round(2).unstack())
print("\n[docs per fold]")
print(df["fold"].value_counts().sort_index().to_dict())



# Token-length profile (does max_length=512 actually bite?) 
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
# add_special_tokens=True counts [CLS]/[SEP] as in real training
tok_lens = df["text_bert"].apply(lambda t: len(tok(t, add_special_tokens=True)["input_ids"]))
df["_tok_len"] = tok_lens

print("\n[token length] describe")
print(tok_lens.describe().round(1))
for thr in (128, 256, 384, 512):
    share = (tok_lens > thr).mean()
    print(f"  > {thr} tokens: {share:.1%} of docs")

[shape] 624 docs, 5 cols
[NaN]
article_id      0
topic_id        0
binary_label    0
fold            0
text_bert       0
dtype: int64
[dup article_id] 0
[folds] [0, 1, 2, 3, 4]

[label balance overall]
{1: 0.667, 0: 0.333}

[label balance per fold]
binary_label     0     1
fold                    
0             0.33  0.67
1             0.33  0.67
2             0.33  0.67
3             0.33  0.67
4             0.33  0.67

[docs per fold]
{0: 126, 1: 126, 2: 126, 3: 123, 4: 123}


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors



[token length] describe
count     624.0
mean      248.2
std       179.6
min        37.0
25%       134.5
50%       213.0
75%       325.2
max      1680.0
Name: text_bert, dtype: float64
  > 128 tokens: 76.4% of docs
  > 256 tokens: 39.9% of docs
  > 384 tokens: 15.5% of docs
  > 512 tokens: 5.4% of docs


In [19]:
# How many articles are above 512 tokens?
over = df.loc[df["_tok_len"] > 512, "_tok_len"]
print(over.describe().round(0))
print("quantili:", over.quantile([.5, .75, .9, .95]).round(0).to_dict())


count      34.0
mean      751.0
std       294.0
min       519.0
25%       557.0
50%       620.0
75%       828.0
max      1680.0
Name: _tok_len, dtype: float64
quantili: {0.5: 620.0, 0.75: 828.0, 0.9: 1204.0, 0.95: 1363.0}


In [20]:
# Are the truncated (long) docs concentrated in one class?
long_mask = df["_tok_len"] > 512
print(df.loc[long_mask, "binary_label"].value_counts().to_dict())
print("share of long docs that are class 1:",
      df.loc[long_mask, "binary_label"].mean().round(2))
print("baseline share class 1 overall:", df["binary_label"].mean().round(2))

{1: 27, 0: 7}
share of long docs that are class 1: 0.79
baseline share class 1 overall: 0.67


In [21]:
# Tokenization, dynamic-padding collator

MAX_LEN = 512  # 5% of docs exceed this; head-only truncation (bias concentrated in head)

# Build HF dataset; keep 'fold' for splitting and 'article_id' for OOF tracking
base_cols = ["text_bert", "binary_label", "fold", "article_id"]
ds_full = Dataset.from_pandas(df[base_cols].reset_index(drop=True), preserve_index=False)

def tokenize_batch(batch):
    return tok(batch["text_bert"], truncation=True, max_length=MAX_LEN)


In [22]:
ds_full = ds_full.map(tokenize_batch, batched=True)
ds_full = ds_full.rename_column("binary_label", "labels")

# Dynamic per-batch padding (not fixed 512) -> less wasted compute/memory
collator = DataCollatorWithPadding(tokenizer=tok)

# Columns the model's forward actually consumes; everything else gets stripped
MODEL_COLS = ["input_ids", "attention_mask", "token_type_ids", "labels"]

Map:   0%|          | 0/624 [00:00<?, ? examples/s]

In [23]:
# Per-fold split builder

def make_fold(k):
    """Split for fold k. Returns cleaned train/eval datasets (tensor cols only),
    plus eval article_ids and gold labels (in eval order) for OOF tracking."""
    folds = ds_full["fold"]
    train_idx = [i for i, f in enumerate(folds) if f != k]
    eval_idx  = [i for i, f in enumerate(folds) if f == k]

    train_ds = ds_full.select(train_idx)
    eval_ds  = ds_full.select(eval_idx)

    # Capture identity BEFORE dropping string columns
    eval_ids    = list(eval_ds["article_id"])
    eval_labels = list(eval_ds["labels"])

    # Strip non-model columns (a string col would break the padding collator)
    drop = [c for c in train_ds.column_names if c not in MODEL_COLS]
    train_ds = train_ds.remove_columns(drop)
    eval_ds  = eval_ds.remove_columns(drop)

    return train_ds, eval_ds, eval_ids, eval_labels

In [24]:
# Sanity check on one fold 
tr, ev, ids, y = make_fold(0)
print(f"[fold 0] train={len(tr)}  eval={len(ev)}")
print(f"[fold 0] model cols = {tr.column_names}")
print(f"[fold 0] eval class balance = "
      f"{pd.Series(y).value_counts(normalize=True).round(2).to_dict()}")

[fold 0] train=498  eval=126
[fold 0] model cols = ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
[fold 0] eval class balance = {1: 0.67, 0: 0.33}


In [25]:
#Model, weighted loss, metrics

import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoModelForSequenceClassification, Trainer

FREEZE_LAYERS = 0  # 0 = full fine-tuning (primary). Set e.g. 6 for the frozen ablation.

def build_model(freeze_layers: int = FREEZE_LAYERS):
    # Re-init from pretrained EVERY fold (no weight leakage across folds).
    # Reseed first so the classifier head init is identical across folds.
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    # Optional: freeze embeddings + first N encoder layers (anti-overfit / cheaper)
    if freeze_layers > 0:
        for p in model.bert.embeddings.parameters():
            p.requires_grad = False
        for layer in model.bert.encoder.layer[:freeze_layers]:
            for p in layer.parameters():
                p.requires_grad = False
    return model

def fold_class_weights(train_labels):
    # Inverse-frequency weights from THIS fold's train split only (no global leakage)
    classes = np.array([0, 1])
    w = compute_class_weight("balanced", classes=classes, y=np.array(train_labels))
    return torch.tensor(w, dtype=torch.float)

class WeightedTrainer(Trainer):
    """Trainer with class-weighted CrossEntropy. Weights set per fold before training."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = (self.class_weights.to(logits.device)
                  if self.class_weights is not None else None)
        loss = nn.functional.cross_entropy(logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),  # primary metric
        "accuracy": accuracy_score(labels, preds),
        "f1_class0": f1_score(labels, preds, pos_label=0),      # minority class watch
        "f1_class1": f1_score(labels, preds, pos_label=1),
    }

# ---- Quick check: weights on fold 0's train split ----
_, _, _, _ = make_fold(0)
tr0 = ds_full.select([i for i, f in enumerate(ds_full["fold"]) if f != 0])
w0 = fold_class_weights(tr0["labels"])
print(f"[fold 0] class weights (0,1) = {w0.tolist()}")

[fold 0] class weights (0,1) = [1.5, 0.75]


In [26]:
# =========================================================
# STEP 4 — Training args + per-fold CV loop, OOF prediction accumulation
# Model 02: BERT document-level
# =========================================================
import pandas as pd
from transformers import TrainingArguments

LR = 3e-5
EPOCHS = 6
TRAIN_BS = 16
EVAL_BS = 32
N_FOLDS = 5

def training_args(fold_k: int):
    return TrainingArguments(
        output_dir=f"./bert_m02_fold{fold_k}",
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",      # log only; NOT used for selection (fixed epochs)
        save_strategy="no",         # no checkpoints during CV; keeps disk clean
        logging_strategy="epoch",
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available(),   # mixed precision on the 5070 -> faster
    )

oof_rows = []      # per-doc OOF predictions across all folds
fold_scores = []   # per-fold f1_macro

for k in range(N_FOLDS):
    print(f"\n===== FOLD {k} =====")
    train_ds, eval_ds, eval_ids, eval_labels = make_fold(k)

    model = build_model(FREEZE_LAYERS)                 # fresh weights every fold
    weights = fold_class_weights(train_ds["labels"])   # weights from THIS train split

    trainer = WeightedTrainer(
        model=model,
        args=training_args(k),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        class_weights=weights,
    )
    trainer.train()

    # Predict on held-out fold
    pred = trainer.predict(eval_ds)
    logits = pred.predictions
    preds = logits.argmax(-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()  # P(class 1)

    fold_f1 = f1_score(eval_labels, preds, average="macro")
    fold_scores.append(fold_f1)
    print(f"[fold {k}] f1_macro = {fold_f1:.4f}")

    for aid, y_true, y_pred, p1 in zip(eval_ids, eval_labels, preds, probs):
        oof_rows.append({"article_id": aid, "fold": k,
                         "y_true": int(y_true), "y_pred": int(y_pred),
                         "prob_class1": float(p1)})

    del model, trainer
    torch.cuda.empty_cache()

oof_df = pd.DataFrame(oof_rows)
print(f"\n[CV] f1_macro per fold = {[round(s,4) for s in fold_scores]}")
print(f"[CV] f1_macro mean±std = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"[OOF] {len(oof_df)} predictions accumulated (should equal dataset size)")


===== FOLD 0 =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ra

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Class0,F1 Class1
1,0.703906,0.666762,0.612716,0.722222,0.406780,0.818653
2,0.659919,0.640781,0.519451,0.523810,0.565217,0.473684
3,0.605919,0.596532,0.711554,0.730159,0.638298,0.784810
4,0.486957,0.581408,0.667681,0.690476,0.580645,0.754717
5,0.334992,0.716643,0.625071,0.626984,0.598291,0.651852
6,0.221028,0.729560,0.656402,0.674603,0.577320,0.735484


[fold 0] f1_macro = 0.6564

===== FOLD 1 =====


...

[fold 3] f1_macro = 0.6527

===== FOLD 4 =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ra

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Class0,F1 Class1
1,0.694574,0.682061,0.469652,0.682927,0.133333,0.805970
2,0.655551,0.636279,0.633722,0.699187,0.478873,0.788571
3,0.549586,0.576016,0.645818,0.650407,0.605505,0.686131
4,0.332781,0.798251,0.623225,0.731707,0.421053,0.825397
5,0.238253,0.687945,0.714396,0.756098,0.605263,0.823529
6,0.132354,0.702579,0.748276,0.780488,0.658228,0.838323


[fold 4] f1_macro = 0.7483

[CV] f1_macro per fold = [0.6564, 0.7421, 0.6716, 0.6527, 0.7483]
[CV] f1_macro mean±std = 0.6942 ± 0.0421
[OOF] 624 predictions accumulated (should equal dataset size)


In [27]:
# 1) Guarda com'è fatto davvero l'input di BERT
print(df["text_bert"].iloc[0][:600])
# quanti 'ENT' per documento in media?
print("ENT per doc (media):", df["text_bert"].str.count(r"\bENT\b").mean().round(1))

# 2) Distribuzione delle predizioni OOF: il modello degenera verso una classe?
print(oof_df["y_pred"].value_counts(normalize=True).round(2).to_dict())
print(oof_df.groupby("fold")["y_pred"].mean().round(2).to_dict())  # frazione predetta =1 per fold

McKINNEY, Texas - ENT was sentenced to 35 years in prison Tuesday, just hours after a Texas jury found him guilty of murder in the 2025 killing of ENT, a fellow high school student, at a Dallas-area track meet. Limited time: Save 25% on NBC News subscription Get exclusive reporting, live Q&As and ad-free reading. The verdict, reached in less than three hours and read by Texas District Court Judge ENT, could have carried a maximum of 99 years. ENT was 17 at the time, but Texas law allowed him to be charged as an adult. He is now 19. During the subsequent sentencing phase Tuesday evening, the ju
ENT per doc (media): 5.3
{1: 0.6, 0: 0.4}
{0: 0.56, 1: 0.55, 2: 0.62, 3: 0.59, 4: 0.69}
